Regresión Lineal Multivariante

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
# Variables exógenas que se usarán como features
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
# Features de tiempo para capturar ciclicidades
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# Lags para las features del modelo de predicción
N_LAGS_AQI = 60  # Lags para la variable objetivo (AQI)
N_LAGS_EXO = 60  # Lags para las variables exógenas (contaminantes)

# Horizontes de predicción actualizados
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5 # Splits para validación cruzada cronológica

# --- Rutas (Asegúrate de que coincidan con tu estructura) ---
PROJECT_ROOT_DIR = os.getcwd()

ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')

# Directorios de datos de entrada
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')

# Directorio de salida para las métricas de este modelo
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
# Nuevo nombre para reflejar el modelo multivariante
PREDICTION_METHOD_NAME = 'regresionLinealMultipleLags' 
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares (Modificadas para Multivariante) ---

def create_multivariate_lagged_features(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    """
    Crea features multivariantes: lags del target, lags de exógenas y features de tiempo.
    El target es y_{t+horizon}. Las features son de tiempo t y anteriores.
    """
    df_processed = pd.DataFrame(index=df_full_city_data.index)

    # Lags de la variable objetivo (ej. AQI_t-1, AQI_t-2, ...)
    for i in range(1, n_lags_target + 1):
        df_processed[f'{target_col}_lag{i}'] = df_full_city_data[target_col].shift(i)

    # Lags de variables exógenas
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(1, n_lags_exo + 1):
            df_processed[f'{exo_col}_lag{i}'] = df_full_city_data[exo_col].shift(i)
    
    # Features de tiempo (contemporáneas, t)
    for tf_col in time_features_cols:
        if tf_col in df_full_city_data.columns:
            df_processed[tf_col] = df_full_city_data[tf_col]
            
    # Target para predicción directa: y_{t+horizon}
    df_processed['target_h'] = df_full_city_data[target_col].shift(-horizon)
    
    # Eliminar filas con cualquier NaN generado por los shifts
    df_processed = df_processed.dropna()
    
    if df_processed.empty:
        return pd.DataFrame(), pd.Series(dtype='float64')
    
    y = df_processed['target_h']
    X = df_processed.drop(columns=['target_h'])
    return X, y

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_multi_horizon(df_city_data_input, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizons_list, n_cv_splits):
    """
    Entrena, predice y evalúa un modelo multivariante para múltiples horizontes usando TimeSeriesSplit.
    """
    results_by_horizon = []
    max_lags = max(n_lags_target, n_lags_exo)
    min_len = max_lags + max(horizons_list) + n_cv_splits * 10
    
    if len(df_city_data_input) < min_len:
        print(f"    Datos insuficientes. Mínimo requerido: {min_len}, disponible: {len(df_city_data_input)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon

    for horizon in horizons_list:
        X, y = create_multivariate_lagged_features(
            df_city_data_input, target_col, exo_cols, time_features_cols,
            n_lags_target, n_lags_exo, horizon
        )

        if X.empty or y.empty or len(X) < n_cv_splits * 2:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        horizon_maes, horizon_rmses, horizon_mases = [], [], []

        for fold, (train_index, test_index) in enumerate(tscv.split(X)):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            if X_train.empty or y_train.empty or X_test.empty or y_test.empty or len(y_train) <=1:
                continue

            model = LinearRegression()
            try:
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)

                # Para MASE, usamos la serie original (target) del fold de entrenamiento
                # El índice final en X_train corresponde a una posición en el df_city_data_input original.
                # La serie de entrenamiento para el naive forecast debe terminar ahí.
                original_series_train_end_idx = X_train.index[-1]
                y_train_for_naive = df_city_data_input[target_col].iloc[:original_series_train_end_idx + 1].values
                
                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                mase = calculate_mase(y_test.values, predictions, y_train_for_naive)

                horizon_maes.append(mae)
                horizon_rmses.append(rmse)
                if not np.isnan(mase): horizon_mases.append(mase)
            except Exception as e:
                print(f"        Error en Fold {fold+1} para horizonte {horizon}h: {e}")
                continue
        
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': np.nanmean(horizon_maes) if horizon_maes else np.nan,
            'RMSE': np.nanmean(horizon_rmses) if horizon_rmses else np.nan,
            'MASE': np.nanmean(horizon_mases) if horizon_mases else np.nan
        })
    return results_by_horizon


# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

# 1. Cargar y preprocesar DataFrame Original (Baseline)
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    # Crear features de tiempo
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

if df_original_full is not None:
    print("\n--- Predicción Multivariante con Datos Originales (Baseline) ---")
    for city_name in df_original_full['City'].unique():
        print(f"  Ciudad: {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        # Rellenar NaNs en el baseline para que sea comparable y ejecutable
        for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
            if df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].ffill().bfill()
        if df_city_original.isnull().values.any():
            print(f"    No se pudieron rellenar todos los NaNs en el baseline. Omitiendo ciudad.")
            continue
        
        horizon_metrics = train_predict_eval_multi_horizon(
            df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
            N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Imputation_Method': 'Original_FFill', 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        if horizon_metrics:
            print(f"    Métricas (Baseline, {horizon_metrics[0]['Horizon']}h): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}, MASE={horizon_metrics[0]['MASE']:.4f}")
            if len(horizon_metrics) > 1:
                print(f"    Métricas (Baseline, {horizon_metrics[-1]['Horizon']}h): MAE={horizon_metrics[-1]['MAE']:.4f}, RMSE={horizon_metrics[-1]['RMSE']:.4f}, MASE={horizon_metrics[-1]['MASE']:.4f}")


# 2. Iterar sobre todos los métodos de imputación y sus archivos
print("\n--- Predicción Multivariante con Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]

for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root):
        continue
    
    print(f"\n--- Procesando datos del método de imputación: {method_name} ---")
    
    for city_folder in os.listdir(imputed_csvs_root):
        city_path = os.path.join(imputed_csvs_root, city_folder)
        if not os.path.isdir(city_path): continue
        
        original_city_name = city_folder.replace("_", " ")
        print(f"  Ciudad: {original_city_name}")

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")

            repetition_files = glob.glob(os.path.join(scenario_path, f"*.csv"))
            if not repetition_files: continue

            # Acumulador para promedios por escenario
            scenario_metrics_accumulator = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}

            for rep_file_path in repetition_files:
                file_name = os.path.basename(rep_file_path)
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                
                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    df_rep['Hour'] = df_rep['Date'].dt.hour
                    df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                    df_rep['Month'] = df_rep['Date'].dt.month
                except Exception as e:
                    print(f"        Error cargando {file_name}: {e}. Omitiendo.")
                    continue
                
                df_city_rep = df_rep[df_rep['City'] == original_city_name].copy().reset_index(drop=True)

                # Verificar si aún quedan NaNs (no debería pasar con buenos métodos de imputación)
                if df_city_rep.isnull().values.any():
                    print(f"        ADVERTENCIA: NaNs encontrados en {file_name} DESPUÉS de imputar. Rellenando con ffill/bfill.")
                    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
                        if df_city_rep[col].isnull().any(): df_city_rep[col] = df_city_rep[col].ffill().bfill()
                    if df_city_rep.isnull().values.any(): continue
                
                horizon_metrics_rep = train_predict_eval_multi_horizon(
                    df_city_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
                    N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
                )

                for hm_rep in horizon_metrics_rep:
                    h = hm_rep['Horizon']
                    all_prediction_metrics_detailed.append({
                        'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': h,
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })
                    if not np.isnan(hm_rep.get('MAE')): scenario_metrics_accumulator[h]['maes'].append(hm_rep['MAE'])
                    if not np.isnan(hm_rep.get('RMSE')): scenario_metrics_accumulator[h]['rmses'].append(hm_rep['RMSE'])
                    if not np.isnan(hm_rep.get('MASE')): scenario_metrics_accumulator[h]['mases'].append(hm_rep['MASE'])

            # Calcular y mostrar promedios del escenario
            print(f"      Promedios para Escenario '{scenario_folder}':")
            for h_summary in [PREDICTION_HORIZONS[0], PREDICTION_HORIZONS[-1]]:
                if h_summary in scenario_metrics_accumulator:
                    avg_mae = np.nanmean(scenario_metrics_accumulator[h_summary]['maes'])
                    avg_rmse = np.nanmean(scenario_metrics_accumulator[h_summary]['rmses'])
                    avg_mase = np.nanmean(scenario_metrics_accumulator[h_summary]['mases'])
                    print(f"        Horizonte {h_summary}h: MAE={avg_mae:.4f}, RMSE={avg_rmse:.4f}, MASE={avg_mase:.4f}")


# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)

    # 1. Guardar CSV detallado global
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\nTodas las métricas de predicción detalladas guardadas en: {detailed_output_path}")
    print(df_all_metrics.head())

    # 2. Crear y guardar CSV de métricas promedio por Ciudad y Método de Imputación
    df_summary = df_all_metrics.groupby(
        ['City', 'Imputation_Method', 'Scenario', 'Horizon_h']
    ).agg(
        Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'),
        Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'),
        Count=('MAE', 'count')
    ).reset_index()

    for city_name in df_summary['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary[df_summary['City'] == city_name]
        
        city_metrics_dir = os.path.join(PREDICTION_METRICS_ROOT_DIR, city_name_safe)
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_safe}.csv')
        
        df_city_summary.to_csv(city_summary_path, index=False)
        print(f"  Resumen de métricas para {city_name} guardado en: {city_summary_path}")

    # 3. Crear y guardar CSV de resumen global
    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(
        Global_Avg_MAE=('MAE', 'mean'),
        Global_Avg_RMSE=('RMSE', 'mean'),
        Global_Avg_MASE=('MASE', 'mean')
    ).reset_index()
    
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"\nResumen global de métricas promedio guardado en: {global_summary_path}")
    print(df_global_summary)

script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")